In [1]:
# !pip uninstall -y transformers accelerate peft trl bitsandbytes
!pip install transformers==4.38.2
!pip install accelerate==0.27.2
!pip install peft==0.9.0
!pip install trl==0.7.11
!pip install bitsandbytes datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.7/130.7 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 96.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 95.1 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.7.1
    Uninstalling huggingface_hub-1.7.1:
      Successfully uninstalled huggingface_hub-1.7.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the follow

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto"
)

model.generation_config.max_length = None

def generate(prompt):

    formatted_prompt = f"<|user|>\n{prompt}\n<|assistant|>\n"

    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # return only assistant answer
    return text.split("<|assistant|>")[-1].strip()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [3]:
# print(generate("Define CNN in neural network?"))

In [4]:
# STEP-1: DATA LOADING & PREPROCESSING
# loading dataset emotionalgo from hugging face
!pip install datasets

from datasets import load_dataset

dataset = load_dataset(
    "parquet",
    data_files={
        "train": "train.parquet",
        "validation": "validation.parquet",
        "test": "test.parquet"
    }
)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

In [5]:
# printing dataset
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 43410
    })
    validation: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5426
    })
    test: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5427
    })
})


In [6]:
print(dataset["train"][100])

{'text': 'You just make her sound awesome.', 'labels': [0], 'id': 'eexo7w2'}


In [7]:
def format_example(example):
    prompt = f"User: {example['text']}\nAssistant:"

    response = "I understand how you feel. I'm here to listen."

    return {
        "prompt": prompt,
        "response": response
    }

formatted_dataset = dataset["train"].map(format_example)

Map:   0%|          | 0/43410 [00:00<?, ? examples/s]

In [8]:
print(formatted_dataset[0])

{'text': "My favourite food is anything I didn't have to cook myself.", 'labels': [27], 'id': 'eebbqej', 'prompt': "User: My favourite food is anything I didn't have to cook myself.\nAssistant:", 'response': "I understand how you feel. I'm here to listen."}


In [9]:
# now we have to combine prompt and text
def build_text(example):
    example["text_combined"] = example["prompt"] + " " + example["response"]
    return example

dataset = formatted_dataset.map(build_text)

Map:   0%|          | 0/43410 [00:00<?, ? examples/s]

In [10]:
# now its in the chatbot format
print(dataset[2]["text_combined"])

User: WHY THE FUCK IS BAYLESS ISOING
Assistant: I understand how you feel. I'm here to listen.


In [11]:
# STEP-2
# LOADING TOKENIZER

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
)

tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [12]:
dataset = dataset.remove_columns(["labels"])

In [19]:
# STEP-3: TOKENIZE
def tokenize(example):

    tokens = tokenizer(
        example["text_combined"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

    tokens["labels"] = tokens["input_ids"]
    return tokens

tokenized_dataset = dataset.map(tokenize)

Map:   0%|          | 0/43410 [00:00<?, ? examples/s]

In [14]:
# removing unused columns
tokenized_dataset = tokenized_dataset.remove_columns(
    ["text","labels","prompt","response","text_combined"]
)

In [15]:
# checking tokenized datset
print(tokenized_dataset[1])

{'id': 'ed00q6i', 'input_ids': [1, 4911, 29901, 2567, 565, 540, 947, 1283, 3654, 29892, 14332, 674, 1348, 19066, 2534, 263, 10569, 885, 3973, 292, 411, 2305, 2012, 310, 2869, 7123, 13, 7900, 22137, 29901, 306, 2274, 920, 366, 4459, 29889, 306, 29915, 29885, 1244, 304, 11621, 29889, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 

In [16]:
# STEP-4: OUR MODEL IS ALREADY LOADED
# SETTING UP WITH LORA {MEMORY EFFICIENT TRAINING}

!pip install peft bitsandbytes accelerate trl

from peft import LoraConfig, get_peft_model

config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, config)

In [17]:
# !pip uninstall -y transformers accelerate peft trl bitsandbytes
# !pip install transformers==4.38.2
# !pip install accelerate==0.27.2
# !pip install peft==0.9.0
# !pip install trl==0.7.11
# !pip install bitsandbytes datasets

In [34]:
print(train_split[0])

{'text': 'I would say they do because it was a horrible accident but they were also in the wrong. The military tried to get them to disperse they didnt.', 'id': 'eehq2an', 'prompt': 'User: I would say they do because it was a horrible accident but they were also in the wrong. The military tried to get them to disperse they didnt.\nAssistant:', 'response': "I understand how you feel. I'm here to listen.", 'text_combined': "User: I would say they do because it was a horrible accident but they were also in the wrong. The military tried to get them to disperse they didnt.\nAssistant: I understand how you feel. I'm here to listen.", 'input_ids': [1, 4911, 29901, 306, 723, 1827, 896, 437, 1363, 372, 471, 263, 4029, 11710, 11423, 541, 896, 892, 884, 297, 278, 2743, 29889, 450, 9121, 1898, 304, 679, 963, 304, 766, 546, 344, 896, 28950, 29889, 13, 7900, 22137, 29901, 306, 2274, 920, 366, 4459, 29889, 306, 29915, 29885, 1244, 304, 11621, 29889, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 

In [33]:
# STEP-8: TRAINING
# from transformers import TrainingArguments, Trainer
# from transformers import default_data_collator

# # Split: 90% train, 10% eval
# n_train     = int(0.9 * len(tokenized_dataset))
# train_split = tokenized_dataset.select(range(n_train))
# eval_split  = tokenized_dataset.select(range(n_train, len(tokenized_dataset)))
# print(f'Train: {len(train_split)} samples  |  Eval: {len(eval_split)} samples')

# # Verify labels are present — training will silently fail without them
# assert 'labels' in train_split.column_names, "ERROR: 'labels' column missing! Re-run the tokenize cell."

# training_args = TrainingArguments(
#     output_dir                  = './emotion-chatbot',
#     per_device_train_batch_size = 8,
#     per_device_eval_batch_size  = 8,
#     num_train_epochs            = 3,
#     learning_rate               = 2e-4,
#     warmup_steps                = 10,
#     weight_decay                = 0.01,
#     logging_dir                 = './logs',
#     logging_steps               = 200,
#     evaluation_strategy         = 'epoch',
#     save_strategy               = 'epoch',
#     load_best_model_at_end      = True,
#     fp16                        = False,   # True = faster on GPU; False = CPU only
#     report_to                   = 'none'
# )

# trainer = Trainer(
#     model         = model,
#     args          = training_args,
#     train_dataset = train_split,
#     eval_dataset  = eval_split,
#     data_collator = default_data_collator,  # REQUIRED — prevents tensor-type warnings
# )

# trainer.train()
# print('\nTraining complete!')

from transformers import TrainingArguments, Trainer
from transformers import DataCollatorWithPadding
import torch

# ✅ Shuffle (better generalization)
dataset = tokenized_dataset.shuffle(seed=42)

# ✅ Slightly smaller dataset for faster training
n_train = 10000   # reduced from 12000
n_eval  = 1000

train_split = dataset.select(range(n_train))
eval_split  = dataset.select(range(n_train, n_train + n_eval))

print(f'Train: {len(train_split)} | Eval: {len(eval_split)}')

# ✅ Safety check
assert 'labels' in train_split.column_names, "ERROR: 'labels' column missing!"

# ✅ Enable memory optimization
model.gradient_checkpointing_enable()

# ✅ Clear GPU memory (important)
torch.cuda.empty_cache()

# ✅ Dynamic padding (saves time + memory)
data_collator = DataCollatorWithPadding(tokenizer)

training_args = TrainingArguments(
    output_dir='./emotion-chatbot',

    # 🔥 Balanced config (fast + stable)
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,   # effective batch = 16

    num_train_epochs=2,   # 🔥 reduced from 3 → faster

    # ⚡ GPU boost
    fp16=True,
    bf16=False,  # set True only if A100/H100 GPU

    # ⚡ Data loading
    dataloader_num_workers=2,
    dataloader_pin_memory=True,

    # ⚡ Reduce overhead
    logging_steps=300,   # less logging = faster
    evaluation_strategy='epoch',
    save_strategy='epoch',

    # ⚡ Learning params
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=10,

    # ⚡ Faster optimizer
    optim="adamw_torch_fused",

    # ⚡ Skip unnecessary stuff
    report_to='none',
    disable_tqdm=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_split,
    eval_dataset=eval_split,
    data_collator=data_collator,
)

trainer.train()

print('\nTraining complete!')

Train: 10000 | Eval: 1000


RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [25]:
# STEP-6: SAVE FINE-TUNED MODEL & TOKENIZER
model.save_pretrained('./emotion-chatbot-final')
tokenizer.save_pretrained('./emotion-chatbot-final')
print('Model and tokenizer saved to ./emotion-chatbot-final')


Model and tokenizer saved to ./emotion-chatbot-final


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [ ]:
# STEP-7: LOAD FINE-TUNED MODEL FOR TESTING
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_NAME = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
FINETUNED_PATH = './emotion-chatbot-final'

tokenizer_test = AutoTokenizer.from_pretrained(FINETUNED_PATH)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map='auto',
    torch_dtype=torch.float16
)

# Load LoRA adapter weights
try:
    fine_tuned_model = PeftModel.from_pretrained(base_model, FINETUNED_PATH)
    print('Loaded as PEFT/LoRA adapter model.')
except Exception as e:
    print(f'PEFT load failed ({e}), using base model directly.')
    fine_tuned_model = base_model

fine_tuned_model.eval()
print('Model ready for inference.')



In [ ]:
# STEP-8: INFERENCE / CHAT FUNCTION
def chat(prompt, max_new_tokens=200, temperature=0.7):
    """Generate an empathetic response from the fine-tuned chatbot."""
    formatted = f'<|user|>\n{prompt}\n<|assistant|>\n'
    inputs = tokenizer_test(formatted, return_tensors='pt').to(fine_tuned_model.device)

    with torch.no_grad():
        output_ids = fine_tuned_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer_test.eos_token_id
        )

    full_text = tokenizer_test.decode(output_ids[0], skip_special_tokens=True)
    if '<|assistant|>' in full_text:
        return full_text.split('<|assistant|>')[-1].strip()
    return full_text.strip()


In [ ]:
# STEP-9: TESTING ON SAMPLE EMOTIONAL INPUTS
test_prompts = [
    'I feel so lonely and no one understands me.',
    'I am really happy today, everything went perfectly!',
    'I failed my exam and I feel like a complete failure.',
    'I am scared about my future and what will happen.',
    'I feel angry all the time and I do not know why.',
]

print('=' * 60)
print('Fine-tuned Emotion Chatbot - Test Results')
print('=' * 60)

for prompt in test_prompts:
    response = chat(prompt)
    print(f'\nUser : {prompt}')
    print(f'Bot  : {response}')
    print('-' * 60)


In [ ]:
# STEP-10: EVALUATION - PERPLEXITY ON EVAL SET
import math
from torch.utils.data import DataLoader
from transformers import default_data_collator

eval_dataloader = DataLoader(
    eval_split, batch_size=4, collate_fn=default_data_collator
)

fine_tuned_model.eval()
total_loss = 0.0
num_batches = 0

with torch.no_grad():
    for batch in eval_dataloader:
        input_ids      = batch['input_ids'].to(fine_tuned_model.device)
        attention_mask = batch['attention_mask'].to(fine_tuned_model.device)
        labels         = batch['labels'].to(fine_tuned_model.device)

        outputs = fine_tuned_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        total_loss += outputs.loss.item()
        num_batches += 1

avg_loss   = total_loss / num_batches
perplexity = math.exp(avg_loss)
print(f'Evaluation Loss      : {avg_loss:.4f}')
print(f'Perplexity (Eval Set): {perplexity:.2f}')
print('Lower perplexity = better model. Typical range for small LLMs: 10-50')
